In [577]:
import numpy as np

In [578]:
UP, RIGHT, DOWN, LEFT = 0, 1, 2, 3
DIRS = {UP: (-1, 0), RIGHT: (0, 1), DOWN: (1, 0), LEFT: (0, -1)}
ARROWS = {UP: "↑", RIGHT: "→", DOWN: "↓", LEFT: "←"}

In [579]:
class GridWorld:

    def __init__(
        self,
        rows: int,
        cols: int,
        step_reward: float,
        terminals: dict[tuple[int, int], float],
        walls: set[tuple[int, int]],
        seed = 0,
        noise = 0.0,
    ):
        self.rows = rows
        self.cols = cols
        self.step_reward = step_reward
        self.terminals = terminals
        self.walls = walls
        self.noise = noise
        self.rng = np.random.default_rng(seed)
        self.s2c = [
            (r, c)
            for r in range(self.rows)
            for c in range(self.cols)
            if (r, c) not in self.walls
        ]
        self.nS = len(self.s2c)
        self.nA = 4
        self.c2s = {cell: i for i, cell in enumerate(self.s2c)}
        self.nonterminal = [
            s for s, cell in enumerate(self.s2c) if cell not in self.terminals
        ]

    def seed(self, v: int):
        self.rng = np.random.default_rng(v)

    def reset(self, s=None):
        """start a new episode; returns the start state"""
        if s is None:
            s = int(self.rng.choice(self.nonterminal))
        return s

    def move(self, s: int, a: int):
        cell = self.s2c[s]
        dr, dc = DIRS[a]
        nxt = (cell[0] + dr, cell[1] + dc)
        if (
            not 0 <= nxt[0] < self.rows
            or not 0 <= nxt[1] < self.cols
            or nxt in self.walls
        ):
            nxt = cell
        reward = self.terminals.get(nxt, self.step_reward)
        return self.c2s[nxt], reward             

    def transitions(self, s: int, a: int):
        result = []
        outcomes = {
            a: 1 - self.noise,
            (a + 1) % 4: self.noise / 2,
            (a - 1) % 4: self.noise / 2,
        }
        for act, prob in outcomes.items():
            ns, r = self.move(s, act)
            result.append((prob, ns, r))
        return result

    def sample_step(self, s: int, a: int):
        result = self.transitions(s, a)
        i = self.rng.choice(len(result), p=[p for p, _, _ in result])
        _, ns, r = result[i]
        done = self.s2c[ns] in self.terminals
        return ns, r, done

    def cell_repr(self, r, c):
        cell = (r, c)
        if cell in self.terminals:
            return f'{self.terminals[cell]:+}'
        elif cell in self.walls:
            return '#'
        else:
            return '·'

    def render(self):
        for r in range(self.rows):
            for c in range(self.cols):
                if r == 0 and c == 0:
                    print(" r/c", end="")
                    print(''.join([f'{v:>3} ' for v in range(self.cols)]))
                if c == 0:
                    print(f'{r:>3} ', end="")
                print(f'{self.cell_repr(r, c):>3} ', end="")
            print()

    def __repr__(self):
        return f'''
Grid(
    rows={self.rows},
    cols={self.cols},
    step_reward={self.step_reward},
    terminals={self.terminals},
    walls={self.walls},
    noise={self.noise},
)
'''

In [580]:
env = GridWorld(
    rows=3,
    cols=4,
    step_reward=0,
    terminals={(0, 3): 1, (1, 3): -1},
    walls={(1, 1)},
    noise=0.2,
)
env


Grid(
    rows=3,
    cols=4,
    step_reward=0,
    terminals={(0, 3): 1, (1, 3): -1},
    walls={(1, 1)},
    noise=0.2,
)

In [581]:
env.render()

 r/c  0   1   2   3 
  0   ·   ·   ·  +1 
  1   ·   #   ·  -1 
  2   ·   ·   ·   · 


### value iteration

In [582]:
def read_policy(
    env: GridWorld,
    V: np.ndarray,
    gamma=0.9,
):
    policy = np.zeros((env.nS, env.nA))
    for s in range(env.nS):
        if env.s2c[s] in env.terminals:
            continue
        best_a = int(np.argmax(q_from_v(env, V, s, gamma)))
        policy[s, best_a] = 1.0
    return policy

def render_policy(env: GridWorld, policy: np.ndarray):
    for r in range(env.rows):
        for c in range(env.cols):
            if r == 0 and c == 0:
                print(" r/c", end="")
                print(''.join([f'{v:>3} ' for v in range(env.cols)]))
            if c == 0:
                print(f'{r:>3} ', end="")  

            cell = (r, c)
            if cell in env.c2s:
                if cell in env.terminals:
                    value = f'{env.terminals[cell]:+}'
                else:
                    value = ARROWS[int(np.argmax(policy[env.c2s[(r, c)]]))]
            else:
                value = '#'
            print(f'{value:>3} ', end='')
        print()    

def show_V(env: GridWorld, V: np.ndarray):
    for r in range(env.rows):
        for c in range(env.cols):
            if r == 0 and c == 0:
                print("  r/c", end="")
                print(''.join([f'{v:>4} ' for v in range(env.cols)]))
            if c == 0:
                print(f'{r:>4} ', end="")            
            cell = (r, c)
            if cell in env.c2s:
                value = round(V[env.c2s[(r, c)]], 2)
            else:
                value = '#'
            print(f'{value:>4} ', end="")
        print()

def q_from_v(
    env: GridWorld,
    V: np.ndarray,
    s: int,
    gamma: float,
):
    q = np.zeros(env.nA)
    if env.s2c[s] in env.terminals:
        return q
    for a in range(env.nA):
        for prob, ns, r in env.transitions(s, a):
            q[a] += prob * (r + gamma * V[ns])
    return q

def value_iteration(
    env: GridWorld,
    gamma=0.9,
    theta=1e-6,
    max_iters=1000,
    verbose=0,
):
    V = np.zeros(env.nS)
    policy = np.zeros((env.nS, env.nA))
    if verbose >= 2:
        print('---------- value_iteration ------------')
        print('V init')
        show_V(env, V)
        print('-' * 25)    
    delta = float('inf')
    i = 0
    while delta >= theta and i < max_iters:
        delta = 0.0
        V_old = V.copy()
        for s in range(env.nS):
            if env.s2c[s] in env.terminals:
                continue
            q = q_from_v(env, V_old, s, gamma)
            V[s] = np.max(q)
            best_a = int(np.argmax(q))
            policy[s] = np.zeros(env.nA)
            policy[s, best_a] = 1.0
            delta = max(delta, abs(V[s] - V_old[s]))
        if verbose >= 1:
            print(f"iter {i}: delta={delta:.6f}")
        if verbose >= 2:
            show_V(env, V)
            print('-' * 25)
        i += 1
    if verbose >= 2:
        render_policy(env, policy)
        print('-' * 25)
    converged = delta < theta
    if verbose >= 1:
        if converged:
                print(f'value_iteration converged in {i - 1} iterations')
        else:
            print(f"value_iteration did not converge in {max_iters} iterations")
    return policy, V, converged

In [583]:
policy, V, converged = value_iteration(env, verbose=2)

---------- value_iteration ------------
V init
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 0: delta=0.800000
  r/c   0    1    2    3 
   0  0.0  0.0  0.8  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 1: delta=0.576000
  r/c   0    1    2    3 
   0  0.0 0.58 0.87  0.0 
   1  0.0    # 0.48  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 2: delta=0.414720
  r/c   0    1    2    3 
   0 0.41 0.73 0.92  0.0 
   1  0.0    # 0.57  0.0 
   2  0.0  0.0 0.34  0.0 
-------------------------
iter 3: delta=0.298598
  r/c   0    1    2    3 
   0 0.56  0.8 0.93  0.0 
   1  0.3    # 0.61  0.0 
   2  0.0 0.25 0.41 0.15 
-------------------------
iter 4: delta=0.237199
  r/c   0    1    2    3 
   0 0.65 0.82 0.94  0.0 
   1 0.46    # 0.63  0.0 
   2 0.24 0.34 0.48 0.21 
-------------------------
iter 5: delta=0.145858
  r/c   0    1    2    3 
   0 0.69

In [584]:
policy = read_policy(env, V)
render_policy(env, policy)

 r/c  0   1   2   3 
  0   →   →   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ←   ↑   ← 


### policy iteration

In [585]:
def policy_evaluation(
    env: GridWorld,
    policy: np.ndarray,
    gamma=0.9,
    theta=1e-6,
    max_iters=1000,
    verbose=0,
):
    V = np.zeros(env.nS)
    if verbose >= 2:
        print('---------- policy_evaluation ------------')
        print('V init')
        show_V(env, V)
        print('-' * 25)    
    delta = float('inf')
    i = 0
    while delta >= theta and i < max_iters:
        delta = 0.0
        V_old = V.copy()
        for s in range(env.nS):
            if env.s2c[s] in env.terminals:
                continue
            V[s] = policy[s] @ q_from_v(env, V_old, s, gamma)
            delta = max(delta, abs(V[s] - V_old[s]))
        if verbose >= 1:
            print(f"iter {i}: delta={delta:.6f}")
        if verbose >= 2:
            show_V(env, V)
            print('-' * 25)
        i += 1
    converged = delta < theta
    if verbose >= 1:
        if converged:
            print(f'policy_evaluation converged in {i - 1} iterations')
        else:
            print(f"policy_evaluation did not converge in {max_iters} iterations")
    return V, converged

In [586]:
policy, *_ = value_iteration(env)
V, converged = policy_evaluation(env, policy, verbose=2)

---------- policy_evaluation ------------
V init
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 0: delta=0.800000
  r/c   0    1    2    3 
   0  0.0  0.0  0.8  0.0 
   1  0.0    # -0.1  0.0 
   2  0.0  0.0  0.0 -0.1 
-------------------------
iter 1: delta=0.576000
  r/c   0    1    2    3 
   0  0.0 0.58 0.86  0.0 
   1  0.0    # 0.47  0.0 
   2  0.0  0.0 -0.08 -0.11 
-------------------------
iter 2: delta=0.414720
  r/c   0    1    2    3 
   0 0.41 0.73 0.92  0.0 
   1  0.0    # 0.56  0.0 
   2  0.0  0.0 0.33 -0.17 
-------------------------
iter 3: delta=0.298598
  r/c   0    1    2    3 
   0 0.56 0.79 0.93  0.0 
   1  0.3    # 0.61  0.0 
   2  0.0  0.0 0.39 0.12 
-------------------------
iter 4: delta=0.214991
  r/c   0    1    2    3 
   0 0.65 0.81 0.94  0.0 
   1 0.46    # 0.63  0.0 
   2 0.21  0.0 0.45 0.19 
-------------------------
iter 5: delta=0.154793
  r/c   0    1    2    3 
   0

In [587]:
def policy_iteration(
    env: GridWorld,
    gamma=0.9,
    theta=1e-6,
    max_iters=1000,
    verbose=0,
    pe_max_iters=1000,
    pe_verbose=0,
    policy: np.ndarray = None
):
    if policy is None:
        policy = np.zeros((env.nS, env.nA))
        # all up policy
        policy[:, UP] = 1.0
    V = np.zeros(env.nS)

    if verbose >= 2:
        print('---------- policy_iteration ------------')
        print('policy init')
        render_policy(env, policy)

    i = 0
    pe_converged = True
    changed = None
    while changed != 0 and i < max_iters:
        V, pe_converged = policy_evaluation(
            env, policy, gamma, theta, max_iters=pe_max_iters, verbose=pe_verbose
        )
        if not pe_converged:
            break
        new_policy = read_policy(env, V, gamma)
        changed = int(
            np.count_nonzero(np.argmax(new_policy, axis=1) != np.argmax(policy, axis=1))
        )
        if verbose >= 2:
            print('-' * 25)
            print(f'iter {i}: V')
            show_V(env, V)
            print('-' * 5)  
        if verbose >= 1:
            prefix = f'iter {i}: ' if verbose == 1 else ''
            print(f'{prefix}actions changed = {changed}')
        if verbose >= 2:
            render_policy(env, new_policy)
        policy = new_policy
        i += 1
    converged = changed == 0
    if verbose >= 1:
        print('-' * 25)  
        if not pe_converged:
            print(
                f"policy_iteration did not converge because policy_evaluation did not converge"
            )
        elif converged:
            print(f'policy_iteration converged in {i - 1} iterations')
        else:
            print(f'policy_iteration did not converge in {max_iters} iterations')
    return policy, V, converged    

In [588]:
policy, V, converged = policy_iteration(env, verbose=2)

---------- policy_iteration ------------
policy init
 r/c  0   1   2   3 
  0   ↑   ↑   ↑  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ↑   ↑   ↑ 
-------------------------
iter 0: V
  r/c   0    1    2    3 
   0 0.07 0.15 0.41  0.0 
   1 0.06    # 0.21  0.0 
   2 0.05 0.04 0.08 -0.87 
-----
actions changed = 5
 r/c  0   1   2   3 
  0   →   →   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   →   ↑   ← 
-------------------------
iter 1: V
  r/c   0    1    2    3 
   0 0.72 0.83 0.94  0.0 
   1 0.63    # 0.64  0.0 
   2 0.54 0.46 0.53 0.31 
-----
actions changed = 1
 r/c  0   1   2   3 
  0   →   →   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ←   ↑   ← 
-------------------------
iter 2: V
  r/c   0    1    2    3 
   0 0.72 0.83 0.94  0.0 
   1 0.63    # 0.64  0.0 
   2 0.55 0.48 0.53 0.31 
-----
actions changed = 0
 r/c  0   1   2   3 
  0   →   →   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ←   ↑   ← 
-------------------------
policy_iteration converged in 2 iterations


### analytic_policy_value

In [589]:
def analytic_policy_value(env: GridWorld, policy: np.ndarray, gamma=0.9):
    n = env.nS
    Ppi = np.zeros((n, n))
    rpi = np.zeros(n)
    for s in range(n):
        if env.s2c[s] in env.terminals:
            continue
        for a in range(env.nA):
            pa = policy[s][a]
            for prob, ns, r in env.transitions(s, a):
                Ppi[s, ns] += pa * prob
                rpi[s] += pa * prob * r
    return np.linalg.solve(np.eye(n) - gamma * Ppi, rpi)

In [590]:
policy, V_star, converged = value_iteration(env)
V = analytic_policy_value(env, policy, 0.9)
show_V(env, V)
print('-' * 25)
equal = np.allclose(V, V_star)
print(f'V derived by analytic_policy_value equal to V_star from value_iteration = {equal}')

  r/c   0    1    2    3 
   0 0.72 0.83 0.94  0.0 
   1 0.63    # 0.64  0.0 
   2 0.55 0.48 0.53 0.31 
-------------------------
V derived by analytic_policy_value equal to V_star from value_iteration = True


### sample_step

In [591]:
env.seed(10)
for _ in range(5):
    ns, r, done = env.sample_step(env.c2s[(0, 2)], RIGHT)
    print(env.s2c[ns], r, done)

(0, 2) 0 False
(0, 3) 1 True
(1, 2) 0 False
(0, 3) 1 True
(0, 3) 1 True


In [592]:
def generate_episode(
    env: GridWorld, policy: np.ndarray, max_steps=1000
):
    episode = []
    s = env.reset()
    for _ in range(max_steps):
        a = int(env.rng.choice(env.nA, p=policy[s]))
        ns, r, done = env.sample_step(s, a)
        episode.append((s, r, ns, done))
        s = ns
        if done:
            break
    return episode

In [593]:
episode = generate_episode(env, policy)
for s, r, ns, done in episode:
    print((env.s2c[s], r, env.s2c[ns], done))

((0, 1), 0, (0, 1), False)
((0, 1), 0, (0, 1), False)
((0, 1), 0, (0, 2), False)
((0, 2), 1, (0, 3), True)


In [594]:
env.render()

 r/c  0   1   2   3 
  0   ·   ·   ·  +1 
  1   ·   #   ·  -1 
  2   ·   ·   ·   · 


In [595]:
def returns_from_episode(episode, gamma=0.9):
    G = 0.0
    out = [0.0] * len(episode)
    for t in range(len(episode) - 1, -1, -1):
        _s, r, _ns, _d = episode[t]
        G = r + gamma * G
        out[t] = G
    return out

In [596]:
returns = returns_from_episode(episode)
returns

[0.7290000000000001, 0.81, 0.9, 1.0]

In [597]:
gamma = 0.9
rewards = [r for (_s, r, _ns, _d) in episode]
G0_brute = sum(gamma ** k * rewards[k] for k in range(len(rewards)))
print(G0_brute)

for t in range(len(episode)):
    brute = sum(gamma**(k - t) * rewards[k] for k in range(t, len(rewards)))
    assert np.isclose(returns[t], brute), t

0.7290000000000001


In [598]:
def mc_prediction(
    env: GridWorld,
    policy: np.ndarray,
    num_episodes: int,
    gamma=0.9,
):
    returns_sum = np.zeros(env.nS)
    returns_cnt = np.zeros(env.nS)
    for _ in range(num_episodes):
        ep = generate_episode(env, policy)
        G = returns_from_episode(ep, gamma)
        seen = set()
        for t, (s, _r, _ns, _d) in enumerate(ep):
            if s in seen:
                continue                   
            seen.add(s)
            returns_sum[s] += G[t]
            returns_cnt[s] += 1
    V = np.zeros(env.nS)
    nz = returns_cnt > 0
    V[nz] = returns_sum[nz] / returns_cnt[nz]
    return V

In [599]:
V_mc = mc_prediction(env, policy, num_episodes=1000)
show_V(env, V_mc)

  r/c   0    1    2    3 
   0 0.71 0.82 0.94  0.0 
   1 0.62    #  0.6  0.0 
   2 0.54 0.47  0.5 0.29 


In [600]:
def rms_error(V_hat: np.ndarray, V_true: np.ndarray,
              nonterminal: list[int]) -> float:
    """Root-mean-square error over the NON-terminal states (terminals are trivially
    0 for everyone and would only dilute the metric)."""
    diff = V_hat[nonterminal] - V_true[nonterminal]
    return float(np.sqrt(np.mean(diff ** 2)))

In [601]:
V_true, _ = policy_evaluation(env, policy)
rng = np.random.default_rng(3)
for num_episodes in [100, 500, 2000]:
    V_mc = mc_prediction(env, policy, num_episodes)
    error = rms_error(V_mc, V_true, env.nonterminal)
    print(f'{num_episodes=}, {error=}')

num_episodes=100, error=0.08582281755489576
num_episodes=500, error=0.016013829867426856
num_episodes=2000, error=0.025439106747398365


In [602]:
show_V(env, V_true)

  r/c   0    1    2    3 
   0 0.72 0.83 0.94  0.0 
   1 0.63    # 0.64  0.0 
   2 0.55 0.48 0.53 0.31 


In [603]:
show_V(env, V_mc)

  r/c   0    1    2    3 
   0 0.72 0.82 0.94  0.0 
   1 0.63    # 0.64  0.0 
   2 0.55 0.48 0.57 0.37 


In [604]:
def td0_prediction(
    env: GridWorld,
    policy: np.ndarray,
    num_episodes: int,
    alpha=0.05,
    gamma=0.9,
):
    V = np.zeros(env.nS)
    for _ in range(num_episodes):
        for (s, r, ns, done) in generate_episode(env, policy):
            target = r + gamma * V[ns] * (1.0 - done)
            V[s] += alpha * (target - V[s])
    return V

In [605]:
V_td0 = td0_prediction(env, policy, num_episodes=1000, alpha=0.05)
show_V(env, V_td0)

  r/c   0    1    2    3 
   0 0.71 0.81 0.91  0.0 
   1 0.63    # 0.57  0.0 
   2 0.54 0.47 0.56 0.32 


In [606]:
V_true, _ = policy_evaluation(env, policy)
rng = np.random.default_rng(3)
for num_episodes in [100, 500, 2000]:
    V_mc = mc_prediction(env, policy, num_episodes)
    error = rms_error(V_mc, V_true, env.nonterminal)
    print(f'{num_episodes=}, {error=}')

num_episodes=100, error=0.09632286936358969
num_episodes=500, error=0.03446593082591272
num_episodes=2000, error=0.006247850925525826
